In [1]:
##!pip install gym
!pip install gymnasium

In [2]:
import gymnasium as gym

# Initialize the Taxi-v3 environment
env = gym.make("Taxi-v4")

## Setup Q-table 
The Q-table has dimensions [state_size, action_size]. For the Taxi environment:

• States: There are 500 discrete states, accounting for the taxi’s position, passenger’s location, and destination.
• Actions: There are 6 possible actions:
0. Move south
1. Move north
2. Move east
3. Move west
4. Pickup passenger
5. Drop off passenger

In [3]:
import numpy as np
import random

# Hyperparameters
alpha = 0.9          # Learning rate
gamma = 0.95         # Discount factor
epsilon = 1.0        # Exploration rate (epsilon-greedy)
epsilon_decay = 0.9995
epsilon_min = 0.01
num_episodes = 10000  # Total training episodes
max_steps = 100       # Max steps per episode

source: https://docs.google.com/document/d/18Vd6PRR6SBkERE1QsJoHj9VEBQoeG_CV2pdNp7O5uGg/edit?tab=t.0
* Learning Rate (alpha): Determines to what extent newly acquired information overrides old information.* 
Discount Factor (gamma): Measures the importance of future rewards.* 
Exploration Rate (epsilon): Controls the trade-off between exploration (trying new actions) and exploitation (using known actions).

The Q-table is a two-dimensional array where rows represent states and columns represent actions.

In [4]:
# Initialize Q-table with zeros
action_size = env.action_space.n
state_size = env.observation_space.n
q_table = np.zeros((state_size, action_size))

We use the epsilon-greedy strategy to balance exploration and exploitation.

Epsilon-Greedy Strategy Explained

• Exploration: Allows the agent to discover new states by taking random actions.
• Exploitation: Utilizes the knowledge accumulated in the Q-table to make the best decision.
• Balance: Starting with a high epsilon encourages exploration; decaying epsilon over time shifts focus to exploitation.

In [5]:
def choose_action(state):
    if random.uniform(0, 1) < epsilon:
        # Explore: choose a random action
        return env.action_space.sample()
    else:
        # Exploit: choose the best known action
        return np.argmax(q_table[state, :])

update q-table

• Temporal Difference (TD) Target: The expected reward of the current action plus the discounted future rewards.

• TD Error: The difference between the TD target and the current Q-valu

The update rule adjusts the Q-value towards the TD target:

• Learning Rate ( \alpha ): Controls how much the new information affects the existing Q-value.
• Discount Factor ( \gamma ): Emphasizes the importance of future rewards.e.

In [6]:
def update_q_table(state, action, reward, next_state):
    best_next_action = np.argmax(q_table[next_state, :])
    td_target = reward + gamma * q_table[next_state, best_next_action]
    td_error = td_target - q_table[state, action]
    q_table[state, action] += alpha * td_error

In [7]:
epsilon = max(epsilon_min, epsilon * epsilon_decay)

-----

## training the agent

• Episode Loop: We run multiple episodes to allow the agent to learn from different starting positions.

• Step Loop: For each step in an episode, the agent chooses an action, observes the outcome, and updates the Q-table.

• Epsilon Decay: Gradually reduce exploration over time.

In [9]:
for episode in range(num_episodes):
    state, info = env.reset()
    done = False
    
    for step in range(max_steps):
        action = choose_action(state)
        next_state, reward, terminated, truncated, info = env.step(action)
        
        update_q_table(state, action, reward, next_state)
        
        state = next_state
        
        if done:
            break
    
    # Decay epsilon
    epsilon = max(epsilon_min, epsilon * epsilon_decay)

----

### Testing the trained agent
• Visualization: We set render_mode=’human’ to visualize the agent’s actions.
• Performance: We monitor the total rewards to assess performance.

In [11]:
env = gym.make("Taxi-v4", render_mode='human')

for episode in range(5):
    state,  info  = env.reset()
    done = False
    total_rewards = 0
    
    for step in range(max_steps):
        env.render()
        action = np.argmax(q_table[state, :])
        next_state, reward, terminated, truncated, info = env.step(action)
        
        total_rewards += reward
        state = next_state
        
        if done:
            print(f"Episode: {episode + 1}, Total Reward: {total_rewards}")
            break

env.close()

KeyboardInterrupt: 